# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [1]:
import numpy as np
import pandas as pd
from cr3bp import (
    create_earth_moon_system,
    grid_search_method
)

### Initialize the System / Problem

In [2]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [3]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

In [4]:
# 10km in natural units
print(10e3/em.l_star)

2.6014568158168575e-05


## Run the Optimization Method

In [5]:
dec_var_ranges = [[3.9, 4.0], [3.0, 3.1], [0.05, 0.12], [0.65, 0.75]]

In [6]:
results_df = grid_search_method(em, dec_var_ranges, 2.6e-4, leo_alt_m, lmo_alt_m)

Performing grid search with 20 x 20 x 25 x 30 = 300000 grid points...
Iteration: 5765/300000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.10 rad, TOF=0.66 s => Distance to LMO=-92.89 km, Total Delta_v=6.12 km/s
New optimal found: Delta_v=3.11 km/s, Theta=3.90 rad, Delta_v_angle=0.10 rad, TOF=0.66 s, Distance to LMO=-92.89 km
Iteration: 5766/300000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.10 rad, TOF=0.66 s => Distance to LMO=58.69 km, Total Delta_v=7.74 km/s
Iteration: 5858/300000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.11 rad, TOF=0.67 s => Distance to LMO=-83.24 km, Total Delta_v=6.22 km/s
Iteration: 5859/300000
Grid point satisfies constraint: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.11 rad, TOF=0.67 s => Distance to LMO=-11.41 km, Total Delta_v=7.86 km/s
Error evaluating grid point: Theta=3.90 rad, Delta_v=3.04, Delta_v_angle=0.07 rad, TOF=0.65 s. Skipping.
Error eval

## Results

In [13]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.989474  3.021053       0.064583  0.750000       3.100649   
1   3.963158  3.026316       0.052917  0.698276       3.115814   
2   3.963158  3.026316       0.064583  0.708621       3.119239   
3   3.942105  3.031579       0.067500  0.677586       3.150977   
4   3.952632  3.031579       0.111250  0.736207       3.167740   
..       ...       ...            ...       ...            ...   
93  3.931579  3.026316       0.073333  0.694828       7.634839   
94  3.910526  3.031579       0.085000  0.674138       7.646455   
95  3.957895  3.021053       0.055833  0.715517       7.658170   
96  3.900000  3.036842       0.108333  0.674138       7.668444   
97  3.915789  3.031579       0.058750  0.653448       7.727783   

    distance_to_lmo  
0     -51880.386440  
1      -3990.465018  
2       8818.895689  
3     -43633.506133  
4     -53255.770416  
..              ...  
93     17646.473281  
94    -47863.407363  
95    -94

In [8]:
optimals = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values

In [9]:
np.save("grid_search_results.npy", optimals)
np.save("grid_search_results_df.npy", results_df)

In [10]:
results_df['distance_to_lmo'] = results_df['distance_to_lmo'] * em.l_star

In [11]:
print(results_df)

       theta   delta_v  delta_v_angle       tof  total_delta_v  \
0   3.989474  3.021053       0.064583  0.750000       3.100649   
1   3.963158  3.026316       0.052917  0.698276       3.115814   
2   3.963158  3.026316       0.064583  0.708621       3.119239   
3   3.942105  3.031579       0.067500  0.677586       3.150977   
4   3.952632  3.031579       0.111250  0.736207       3.167740   
..       ...       ...            ...       ...            ...   
93  3.931579  3.026316       0.073333  0.694828       7.634839   
94  3.910526  3.031579       0.085000  0.674138       7.646455   
95  3.957895  3.021053       0.055833  0.715517       7.658170   
96  3.900000  3.036842       0.108333  0.674138       7.668444   
97  3.915789  3.031579       0.058750  0.653448       7.727783   

    distance_to_lmo  
0     -51880.386440  
1      -3990.465018  
2       8818.895689  
3     -43633.506133  
4     -53255.770416  
..              ...  
93     17646.473281  
94    -47863.407363  
95    -94

In [12]:
shrink = 0.5
new_ranges = []
for i in range(4):
    current_span = dec_var_ranges[i][1] - dec_var_ranges[i][0]
    half_width = current_span * shrink / 2
    new_min = max(dec_var_ranges[i][0], optimals[i] - half_width)
    new_max = min(dec_var_ranges[i][1], optimals[i] + half_width)
    new_ranges.append([new_min, new_max])
print(new_ranges)

[[np.float64(3.9644736842105264), 4.0], [3.0, np.float64(3.0460526315789473)], [0.05, np.float64(0.08208333333333334)], [np.float64(0.725), 0.75]]
